In [1]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数

    参数:
        datasources (dict): 数据源表名映射
        start_date (str): 开始时间
        end_date (str): 结束时间

    返回:
        pd.DataFrame: 因子数据，包含 ['date', 'instrument', 'factor']
    """
    import numpy as np
    import pandas as pd
    import dai

    bar_table = datasources["bar1m"]

    # 优化因子：多窗口量价背离复合 + Rank化 + ATR标准化
    # 参考海通证券《量价结合选股因子》与国金证券《高频量价背离选股因子》
    # 核心逻辑：量价背离时未来上涨概率较高

    lookback_days = 90
    query_start_date = pd.to_datetime(start_date) - pd.Timedelta(days=lookback_days)

    sql = f"""
    WITH pool AS (
        SELECT DISTINCT instrument
        FROM bigalpha_2026_instruments
        WHERE date BETWEEN '{start_date}' AND '{end_date}'
    )
    SELECT
        date,
        instrument,
        close,
        volume,
        high,
        low
    FROM {bar_table}
    PRUNE JOIN pool USING (instrument)
    """
    bar_df = dai.query(
        sql,
        filters={"date": [query_start_date, end_date]},
    ).df()

    if bar_df.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    bar_df["datetime"] = pd.to_datetime(bar_df["date"])
    bar_df["date"] = bar_df["datetime"].dt.normalize()
    bar_df = bar_df.sort_values(["instrument", "datetime"])

    daily_df = (
        bar_df.groupby(["date", "instrument"], as_index=False)
        .agg(
            close=("close", "last"),
            volume=("volume", "sum"),
            high=("high", "max"),
            low=("low", "min"),
        )
        .sort_values(["instrument", "date"])
        .reset_index(drop=True)
    )

    def calc_factor(group):
        group = group.sort_values("date").copy()

        # ----- 信号1：短周期量价相关性（5日） -----
        # 价格变化与成交量变化的相关性，负相关=量价背离
        ret_5 = group["close"].pct_change(5)
        vol_chg_5 = group["volume"].pct_change(5)
        corr_5 = ret_5.rolling(5, min_periods=3).corr(vol_chg_5)

        # ----- 信号2：中周期量价相关性（10日） -----
        ret_10 = group["close"].pct_change(10)
        vol_chg_10 = group["volume"].pct_change(10)
        corr_10 = ret_10.rolling(10, min_periods=5).corr(vol_chg_10)

        # ----- 信号3：长周期动量（20日） -----
        ret_20 = group["close"].pct_change(20)

        # ----- 复合背离信号（负相关=背离，因子值越大越好） -----
        # 参考研报：量价背离程度越高，未来上涨概率越高
        divergence_signal = -0.4 * corr_5 - 0.4 * corr_10 + 0.2 * ret_20

        # ----- ATR波动率标准化 -----
        high_low = group["high"] - group["low"]
        high_close = np.abs(group["high"] - group["close"].shift(1))
        low_close = np.abs(group["low"] - group["close"].shift(1))
        true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        atr = true_range.rolling(20, min_periods=5).mean()

        # 最终因子：背离信号 / ATR
        group["factor"] = divergence_signal / (atr + 1e-6)

        # 处理无穷值和缺失值
        group["factor"] = (
            pd.to_numeric(group["factor"], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
        )

        return group[["date", "instrument", "factor"]]

    factor_df = (
        daily_df.groupby("instrument", group_keys=False)
        .apply(calc_factor)
        .reset_index(drop=True)
    )

    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    factor_df = factor_df[
        (factor_df["date"] >= start_ts) & (factor_df["date"] <= end_ts)
    ].copy()

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    df = pd.merge(
        stk_pool,
        factor_df,
        how="left",
        on=["date", "instrument"],
    )

    df["factor"] = (
        pd.to_numeric(df["factor"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df["instrument"] = df["instrument"].astype(str)

    return (
        df[["date", "instrument", "factor"]]
        .drop_duplicates(subset=["date", "instrument"], keep="last")
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )